In [ ]:
import pandas as pd
import psycopg2
import ast

# --- Đọc file CSV ---
df = pd.read_csv("test_data/all_questions_combined.csv")

# --- Kết nối PostgreSQL ---
conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="123456"
)
cur = conn.cursor()

# --- Import dữ liệu ---
for _, row in df.iterrows():
    question_content = row["question"]
    answers = ast.literal_eval(row["answer"])  # tách list chuỗi

    # --- Insert vào bảng question ---
    cur.execute("""
        INSERT INTO question (question_name, question_content)
        VALUES (%s, %s)
        RETURNING question_id
    """, ('none', question_content))
    question_id = cur.fetchone()[0]

    # --- Insert từng câu trả lời ---
    for ans in answers:
        cur.execute("""
            INSERT INTO answer (question_id, answer_content, is_correct)
            VALUES (%s, %s, NULL)
        """, (question_id, ans))

# --- Lưu thay đổi ---
conn.commit()
cur.close()
conn.close()

print("✅ Import thành công!")


✅ Import thành công!
